# 課題と解答例：30_agent_foundations

元Notebook: [../30_agent_foundations.ipynb](../30_agent_foundations.ipynb)

## 課題

1. 逐次更新の順序をA，Bの順へ変え，結果が変わる理由を説明する．
2. AとBの間に空きセルを1つ置き，2種類の更新結果が一致するか確認する．
3. trialsを10，100，1000と変え，Aの勝率が0.5へどのように近づくか記録する．
4. seedだけを変えた場合と，Aを選ぶ確率を0.7へ変えた場合の意味の違いを説明する．
5. 各更新後に人数保存と1セル1人を検査するassert文を追加する．

### 追加実装課題

6. `compare_update_orders(initial_positions, orders)` を作り，複数の逐次更新順序の結果を表にする．
7. `run_conflict_trials(trials_list, p_A, seed)` を作り，試行回数ごとのAの勝率をDataFrameで返す．
8. `check_invariants(positions)` を作り，人数保存と1セル1人を `assert` で検査する．逐次更新後と一斉更新後の両方で実行する．

## 解答例

1. A，Bの順では，Aは右隣がBで埋まっているため動けない．その後Bが右へ動く．B，Aの順ならBが先に右へ動き，空いた場所へAが入れる．同じ移動規則でも処理順序で結果が変わる．

2. AとBの間に空きセルを1つ置くと，競合が起きないため，逐次更新でも一斉更新でも結果は一致する．

3. `trials` を増やすと，Aの勝率は0.5に近づく．ただし有限回では完全に0.5にはならない．

4. seedだけを変える場合は同じモデルでの乱数ばらつきを見る実験である．Aを選ぶ確率を0.7に変える場合は，競合解決規則そのものを変える実験である．

5. 不変量の検査例である．

   ```python
   assert set(new_positions) == {'A', 'B'}
   assert len(set(new_positions.values())) == len(new_positions)
   ```

6. 更新順序比較の例である．

   ```python
   def compare_update_orders(initial_positions, orders):
       rows = []
       for order in orders:
           result = sequential_step(initial_positions, order=order)
           rows.append({"order": order, "A": result["A"], "B": result["B"]})
       return pd.DataFrame(rows)
   ```

7. 乱数競合の表は次のように作れる．

   ```python
   def run_conflict_trials(trials_list, p_A=0.5, seed=0):
       rng = np.random.default_rng(seed)
       rows = []
       for trials in trials_list:
           winners = rng.choice(["A", "B"], size=trials, p=[p_A, 1-p_A])
           rows.append({"trials": trials, "p_A": p_A, "A_win_rate": np.mean(winners == "A")})
       return pd.DataFrame(rows)
   ```

8. `check_invariants` は位置の重複を検査する．

   ```python
   def check_invariants(positions):
       assert set(positions) == {"A", "B"}
       assert len(set(positions.values())) == len(positions)
   ```